# Chapter 5: The Digital Canvas
## Companion Notebook — "Eyes of the Machine"

Learn to manipulate images using OpenCV — blur, sharpen, resize, rotate, crop, and create your own filters.

---
### 5A: Setup — Imports for Colab
These imports must run first.

In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

---
### 5B: Load an Image with OpenCV
Upload a photo. OpenCV stores it in BGR order (historical quirk).

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

img_bgr = cv.imread(filename)
print("Shape:", img_bgr.shape)
print("Data type:", img_bgr.dtype)

print("Displayed with OpenCV (BGR is correct for OpenCV):")
cv2_imshow(img_bgr)

---
### 5C: BGR → RGB + Grayscale
Convert to RGB for matplotlib display, and to grayscale for brightness-only.

In [ ]:
img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("RGB (correct colors)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("Grayscale (brightness only)")
plt.axis('off')
plt.show()

---
### 5D: Blur — Making the Image Soft
GaussianBlur averages each pixel with its neighbors. Bigger kernel = more blur.

In [ ]:
blurred_light = cv.GaussianBlur(img_rgb, (5, 5), 0)
blurred_heavy = cv.GaussianBlur(img_rgb, (31, 31), 0)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(blurred_light)
plt.title("Blur (kernel=5)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(blurred_heavy)
plt.title("Blur (kernel=31)")
plt.axis('off')
plt.show()

---
### 5E: Sharpen — Making the Image Crisp
A custom 3x3 kernel emphasizes differences between neighboring pixels.

In [ ]:
kernel_sharpen = np.array([
    [0, -1,  0],
    [-1,  5, -1],
    [0, -1,  0]
])

sharpened = cv.filter2D(img_rgb, -1, kernel_sharpen)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(sharpened)
plt.title("Sharpened")
plt.axis('off')
plt.show()

---
### 5F: Resize and Rotate
Change dimensions and spin the image.

In [ ]:
height, width = img_rgb.shape[:2]
new_width = width // 2
new_height = height // 2

small_img = cv.resize(img_rgb, (new_width, new_height))
print(f"Original: {width}x{height} -> Resized: {new_width}x{new_height}")

plt.imshow(small_img)
plt.title("Half size")
plt.axis('off')
plt.show()

# Rotate 45 degrees.
center = (width // 2, height // 2)
rotation_matrix = cv.getRotationMatrix2D(center, 45, 1.0)
rotated = cv.warpAffine(img_rgb, rotation_matrix, (width, height))

plt.imshow(rotated)
plt.title("Rotated 45 degrees")
plt.axis('off')
plt.show()

---
### 5G: Crop — Keep Only the Interesting Part
Numpy array slicing = cropping. Rows (y) come first!

In [ ]:
crop_y_start = height // 3
crop_y_end = 2 * height // 3
crop_x_start = width // 3
crop_x_end = 2 * width // 3

cropped = img_rgb[crop_y_start:crop_y_end, crop_x_start:crop_x_end]

plt.imshow(cropped)
plt.title("Cropped center region")
plt.axis('off')
plt.show()

---
### 5H: Vintage Warm Color Filter
Manipulate color channels directly to create a vintage look.

In [ ]:
filtered = img_rgb.astype(np.float32).copy()
filtered[:, :, 0] *= 0.8   # Reduce red
filtered[:, :, 2] *= 1.3   # Increase blue
filtered = np.clip(filtered, 0, 255).astype(np.uint8)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(filtered)
plt.title("Warm Vintage Filter")
plt.axis('off')
plt.show()

cv.imwrite("filtered_photo.jpg", cv.cvtColor(filtered, cv.COLOR_RGB2BGR))
print("Saved as filtered_photo.jpg")

---
## 🧠 Challenge: The "8-Hours-of-Sleep" Filter

Upload a tired-face selfie. Write a filter pipeline that makes you look well-rested:
1. Brighten the image (+30 to all pixels).
2. Reduce redness in skin (multiply red channel by 0.8).
3. Mild blur for smooth skin (kernel 3x3).
4. Increase contrast (stretch values).
5. Show before and after side by side.

In [ ]:
# --- YOUR EXPERIMENT ---
face = img_rgb.astype(np.float32).copy()

# Step 1: Brighten
face = face + 30

# Step 2: Reduce red
face[:, :, 0] *= 0.8

# Step 3: Mild blur
face = cv.GaussianBlur(face, (3, 3), 0)

# Step 4: Increase contrast
# (stretch the darkest pixels toward 0 and brightest toward 255)
min_val = np.min(face)
max_val = np.max(face)
face = (face - min_val) / (max_val - min_val) * 255

# Clip and convert back.
face = np.clip(face, 0, 255).astype(np.uint8)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Before (Tired)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(face)
plt.title("After (Rested)")
plt.axis('off')
plt.show()

print("Filter applied! Share with a friend and ask which looks more rested.")